# Project — Fuel prices — *Prix des carburants en France, flux instantané v2*
**MSA-DATI07-03 · Advanced EDA & Data Collection** · difficulty ⭐ easy

This is your **project notebook**. It already follows the six-section structure from Session 6 and contains working collection code for this API. Your job: fill every `# TODO` and every *TODO* markdown cell, on **real data you collected yourselves**.

| # | Section | Answers the question… |
|---|---|---|
| 1 | Context & provenance | Where does this data come from, and what question is it for? |
| 2 | Setup & parameters | Can someone else run this and get the same thing? |
| 3 | Collection, ingestion & schema | Did we collect it properly, and is it what we think it is? |
| 4 | Data quality | What is missing, duplicated or weird — and what did we decide? |
| 5 | Exploration | What does the data say? Every figure captioned. |
| 6 | Findings & next steps | What can a modeller trust, and what should they test? |

**Deliverables:** this notebook (Restart & Run All passes, same fingerprint twice) · `requirements.txt` · the 2-page data-quality report (`DATA_QUALITY_REPORT_TEMPLATE.md`) · regular Git commits.

**Possible questions** (pick one or write your own — one per group):
- How much do fuel prices vary between regions, and between motorway and other stations?
- How stale are the published prices — which stations have not updated for weeks?
- Which fuels are missing where (E85, GPLc), and is that a data gap or a real absence?

---
## 1 · Context & provenance

| Field | Your answer |
|---|---|
| **Business question** | *Proposition à valider par le groupe :* quelle est la fraîcheur des prix publiés, et quelles stations n'ont pas mis à jour leurs prix depuis plusieurs semaines ? |
| **Source** | Ministère de l'Économie open-data portal (data.economie.gouv.fr), dataset `prix-des-carburants-en-france-flux-instantane-v2` |
| **Dataset page / API docs** | https://www.data.economie.gouv.fr/explore/dataset/prix-des-carburants-en-france-flux-instantane-v2/ |
| **Collected by / when** | *Membres du groupe : à renseigner.* Date et heure de collecte : `collected_at` (cellule 3.2), à reporter ici. |
| **Collection method** | REST API (Opendatasoft Explore v2.1), pagination par offset de 100, cache `requests-cache` 7 jours (Section 3) |
| **Filters applied** | Aucun : `WHERE = None`, France entière, tri par `id` (voir cellule de paramètres) |
| **Licence / constraints** | Licence Ouverte / Etalab 2.0, selon le notebook. *À vérifier sur la page du dataset avant rendu.* |
| **Unit of observation** | Une ligne = une station-service à l'instant de la collecte. *Confirmé sur le snapshot CSV de référence : 9 824 lignes, 9 824 `id` distincts, aucun doublon exact. À reconfirmer sur votre collecte API (cellule 3.2b).* |
| **Known limitations** | (1) **Snapshot** : le contenu change à chaque appel, deux collectes ne donnent pas les mêmes lignes. (2) **Plafond API** : l'API refuse offset + limit > 10 000, donc au-delà de 10 000 stations la collecte est tronquée (contrôle en 3.2b). (3) **Dates de mise à jour par carburant** : un prix peut être ancien même si la station est active. (4) **Carburant absent** : une station ne vend pas forcément tous les carburants ; l'API ne dit pas toujours si c'est une absence réelle ou un défaut de déclaration. (5) **Pas d'historique** : seul l'état courant est disponible. (6) **Coordonnées** : dans l'export, `latitude` et `longitude` sont en Lambert 93 (mètres) ; les degrés WGS84 sont dans `geom`. |

**Questions possibles** (une par groupe) :
- Quelle est la variation des prix entre régions, et entre stations d'autoroute et les autres ?
- Quelle est la fraîcheur des prix publiés : quelles stations n'ont pas mis à jour leurs prix depuis plusieurs semaines ?
- Quels carburants manquent où (E85, GPLc), et s'agit-il d'un défaut de données ou d'une absence réelle ?


---
## 2 · Setup & parameters
On Colab, run the install cell once per session.

In [1]:
# Colab / fresh environment only
import importlib.util
missing = [p for p in ("pandera", "requests_cache") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q "pandera>=0.24" requests-cache

In [2]:
# --- Parameters: the ONLY cell you should need to edit to rerun on new data ---
SEED = 42
DATA_DIR = "data"
REPORT_DIR = "reports"
CACHE_HOURS = 24 * 7                              # re-use API responses for a week
IQR_K = 1.5
MAX_MISSING_SHARE = 0.30
PRICE_MIN, PRICE_MAX = 0.5, 3.5                  # EUR per litre: outside = typo or error
FR_LAT, FR_LON = (41.0, 51.5), (-5.5, 9.8)       # metropolitan France incl. Corsica

BASE_URL = "https://data.economie.gouv.fr"
DATASET_ID = "prix-des-carburants-en-france-flux-instantane-v2"
WHERE = None                                      # whole of France is just under the 10,000-row cap
ORDER_BY = "id"
MAX_ROWS = 10_000
PAUSE_S = 0.5

In [3]:
import hashlib, json, random, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandera
import pandera.pandas as pa
import requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(SEED); np.random.seed(SEED)
DATA, REPORTS = Path(DATA_DIR), Path(REPORT_DIR)
for p in (DATA / "raw", DATA / "processed", REPORTS):
    p.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

for lib in (pd, np, sns, pandera, requests):
    print(f"{lib.__name__:<10}{lib.__version__}")

pandas    2.3.3
numpy     2.2.6
seaborn   0.13.2
pandera   0.35.0
requests  2.32.5


In [4]:
# Write requirements.txt from the packages this notebook imports
from importlib.metadata import version
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pandera", "pyarrow", "requests", "requests-cache", "urllib3"]
Path("requirements.txt").write_text("".join(f"{p}=={version(p)}\n" for p in pkgs))
print(Path("requirements.txt").read_text())

pandas==2.3.3
numpy==2.2.6
matplotlib==3.10.9
seaborn==0.13.2
pandera==0.35.0
pyarrow==25.0.1
requests==2.32.5
requests-cache==1.3.3
urllib3==2.5.0



---
## 3 · Collection, ingestion & schema

### 3.1 A polite, cached HTTP session
- **Cache**: the same request is never sent twice (fast reruns, kind to the API).
- **Retry with backoff** on 429 / 5xx, respecting `Retry-After`.
- **Throttle**: pause between requests that actually hit the network.

In [5]:
Path("cache").mkdir(parents=True, exist_ok=True)   # requests_cache writes its SQLite file here
session = requests_cache.CachedSession("cache/api", expire_after=CACHE_HOURS * 3600)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True)))
session.headers["User-Agent"] = "MSA-DATI07-student-project/1.0"

def get(url, params=None):
    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    if not getattr(r, "from_cache", False):
        time.sleep(PAUSE_S)
    return r


### 3.2 Collect

In [6]:
API_TOTAL = None   # filled by collect_ods, checked in 3.2b

def collect_ods(base_url, dataset_id, where=None, order_by=None, max_rows=MAX_ROWS, use_export=False):
    global API_TOTAL                                 # so that the check in 3.2b sees the value
    """Opendatasoft Explore API v2.1. Offset pagination, 100 rows per page.
    The API refuses offset + limit > 10,000: above that, filter with `where` or use the export endpoint."""
    url = f"{base_url}/api/explore/v2.1/catalog/datasets/{dataset_id}"
    params = {k: v for k, v in {"where": where, "order_by": order_by}.items() if v}
    if use_export:                                   # one call, no row cap, returns a JSON list
        r = get(f"{url}/exports/json", params=params)
        return r.json()[:max_rows]
    rows, offset, limit = [], 0, 100
    while offset < min(max_rows, 10_000):
        r = get(f"{url}/records", params={**params, "limit": limit, "offset": offset})
        page = r.json()
        if offset == 0:
            API_TOTAL = page["total_count"]
            print(f"API reports {API_TOTAL:,} matching rows")
        rows += page["results"]
        if len(page["results"]) < limit:             # last page
            break
        offset += limit
    return rows[:max_rows]

In [7]:
collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
raw_rows = collect_ods(BASE_URL, DATASET_ID, where=WHERE, order_by=ORDER_BY)
print(f"collected {len(raw_rows):,} rows at {collected_at}")

API reports 9,826 matching rows
collected 9,826 rows at 2026-11-05T16:48:30+00:00


### 3.2b Collection checks
Three checks before anything is saved: the collection is complete (not cut by the 10,000 cap), `id` identifies a station, and the rows are not empty.

In [8]:
# Collection checks: stop early rather than analyse a truncated or broken collection
expected = min(API_TOTAL, MAX_ROWS) if API_TOTAL is not None else None
print(f"API total: {API_TOTAL} · collected: {len(raw_rows)} · expected: {expected}")
if API_TOTAL is not None and API_TOTAL > MAX_ROWS:
    print("⚠️ API total is above MAX_ROWS: the collection is truncated. Filter with WHERE (e.g. by region) and collect again.")
if expected is not None and len(raw_rows) != expected:
    print("⚠️ collected rows differ from the expected number: check the pagination before going on.")
assert len(raw_rows) > 0, "no rows collected: tell the teacher and switch dataset"
ids = [r.get("id") for r in raw_rows]
print(f"distinct ids: {len(set(ids)):,} / {len(ids):,}")


API total: 9826 · collected: 9826 · expected: 9826
distinct ids: 9,826 / 9,826


### 3.3 Flatten, save raw Parquet + README

In [ ]:
df_raw = pd.json_normalize(raw_rows, sep='_')

In [ ]:
raw_path = DATA / "raw" / "raw.parquet"
df_raw.to_parquet(raw_path, index=False)
(DATA / "raw" / "README.md").write_text(f"""# Raw data
- Source: see Section 1 of the notebook
- Dataset: {DATASET_ID} ({BASE_URL})
- Collected at (UTC): {collected_at}
- API total_count at collection: {API_TOTAL}
- Filter (WHERE): {WHERE}
- Rows x columns: {df_raw.shape[0]} x {df_raw.shape[1]}
- Columns: {", ".join(df_raw.columns)}
""")
print(df_raw.shape)
df_raw.head()

In [ ]:
# First look: types, missing share, number of distinct values per column
overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "share_missing": df_raw.isna().mean().round(3),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("share_missing", ascending=False)
overview

### 3.5 First look at update dates (for Section 4)
Per-fuel age of the last price update, measured against the collection time. Only a first look: the decisions come in Section 4.

In [ ]:
# Age in days of each price, against the collection time (UTC)
ref = pd.Timestamp(collected_at)
for c in [c for c in df_raw.columns if "mis à jour" in c.lower() or c.endswith("_maj")]:
    ts = pd.to_datetime(df_raw[c], errors="coerce", utc=True)
    age = (ref - ts).dt.days
    print(f"{c:<35} median age: {age.median():>6} d · >7 d: {(age > 7).mean():.1%} · >30 d: {(age > 30).mean():.1%}")


### 3.4 Schema-as-contract

Columns are kept under the names used in the rest of the notebook (`RENAME`). The contract checks every kept column, with `strict=True` (an unexpected column fails), six domain checks, and a justification for every `nullable=True` in its `description`.

**Check the names first.** The API field names in `RENAME` (for example `gazole_prix`, `geom_lat`) are the expected names, not verified ones. The next cell stops with the list of names it cannot find: copy the exact names from the printed `df_raw.columns` and fix `RENAME`.

**Group task:** read every `nullable=True` and its description, and change it if your group disagrees.


In [ ]:
# Exact column names from the API (copy from this list into RENAME, do not guess)
print(list(df_raw.columns))

# --- 3.4a Column selection (the selection itself is a decision: it is logged below) ---
# Left: field name in df_raw. Right: name used in this notebook.
RENAME = {
    "id": "id",
    "cp": "cp",
    "ville": "ville",
    "adresse": "adresse",
    "departement": "departement",
    "region": "region",
    "geom_lat": "lat",               # WGS84 degrees. NOT 'latitude': Lambert 93 metres in the export
    "geom_lon": "lon",
    "gazole_prix": "prix_gazole", "gazole_maj": "maj_gazole",
    "sp95_prix": "prix_sp95",     "sp95_maj": "maj_sp95",
    "e85_prix": "prix_e85",       "e85_maj": "maj_e85",
    "gplc_prix": "prix_gplc",     "gplc_maj": "maj_gplc",
    "e10_prix": "prix_e10",       "e10_maj": "maj_e10",
    "sp98_prix": "prix_sp98",     "sp98_maj": "maj_sp98",
}
missing = [c for c in RENAME if c not in df_raw.columns]
if missing:
    raise KeyError(f"Not in df_raw.columns: {missing}. Copy the exact names from the list printed above and fix RENAME.")

KEEP = list(RENAME)
dropped = [c for c in df_raw.columns if c not in KEEP]
df_sel = df_raw[KEEP].rename(columns=RENAME).copy()
log("(all raw columns)", f"{len(dropped)} raw columns not kept", len(df_raw), "dropped",
    "Not needed for the questions or the contract: nested JSON (prix, services, horaires), "
    "rupture details, and administrative codes duplicated by region/departement names.")

# --- 3.4b Types: what the schema expects ---
FUELS = ["gazole", "sp95", "e85", "gplc", "e10", "sp98"]
df_sel["id"] = df_sel["id"].astype("int64")
df_sel["lat"] = pd.to_numeric(df_sel["lat"], errors="coerce")
df_sel["lon"] = pd.to_numeric(df_sel["lon"], errors="coerce")
for f in FUELS:
    df_sel[f"prix_{f}"] = pd.to_numeric(df_sel[f"prix_{f}"], errors="coerce")
    df_sel[f"maj_{f}"] = pd.to_datetime(df_sel[f"maj_{f}"], utc=True, errors="coerce").dt.tz_localize(None)

# --- 3.4c Contract ---
COLLECTED_NAIVE = pd.Timestamp(collected_at).tz_convert("UTC").tz_localize(None)

def in_range(lo, hi, name):
    return pa.Check.in_range(lo, hi, include_min=True, include_max=True, name=name)

price_cols = {
    f"prix_{f}": pa.Column(float, nullable=True,
        checks=in_range(PRICE_MIN, PRICE_MAX, "price in plausible range"),
        description="nullable: the station does not sell this fuel, or did not report it")
    for f in FUELS}
maj_cols = {
    f"maj_{f}": pa.Column(pa.DateTime, nullable=True,
        checks=pa.Check(lambda s: s.dropna() <= COLLECTED_NAIVE, name="update not in the future"),
        description="nullable: empty when the matching price is empty")
    for f in FUELS}

schema_raw = pa.DataFrameSchema({
    "id": pa.Column("int64", unique=True, nullable=False, checks=pa.Check.gt(0)),
    "cp": pa.Column(str, nullable=False, checks=pa.Check.str_matches(r"^\d{5}$", name="postal code: 5 digits")),
    "ville": pa.Column(str, nullable=True, description="nullable: 4 rows of the snapshot have no city; kept, not dropped"),
    "adresse": pa.Column(str, nullable=False),
    "departement": pa.Column(str, nullable=True, description="nullable: same 4 rows as ville"),
    "region": pa.Column(str, nullable=True, description="nullable: same 4 rows as ville"),
    "lat": pa.Column(float, nullable=False, checks=in_range(*FR_LAT, "latitude inside France")),
    "lon": pa.Column(float, nullable=False, checks=in_range(*FR_LON, "longitude inside France")),
    **price_cols, **maj_cols,
}, strict=True)

# --- 3.4d Validate with lazy=True and keep the failure summary for the report ---
try:
    schema_raw.validate(df_sel, lazy=True)
    failures = pd.DataFrame(columns=["column", "check", "failure_case"])
    print("✅ raw data passes the contract", df_sel.shape)
except pa.errors.SchemaErrors as err:
    failures = err.failure_cases[["column", "check", "failure_case"]]
    print(f"❌ {len(failures)} violations: summary below, also saved in reports/")
    display(failures.fillna("(table-level)").groupby(["column", "check"]).size().rename("n").reset_index())
failures.to_csv(REPORTS / "schema_failures_raw.csv", index=False)

---
## 4 · Data quality — measure → decide → log

Every decision gets one `log(...)` call. The table at the end of this section goes into your report.

**What to look for in this dataset:**
- Price update dates: how many prices are older than 1 week, 1 month? Are stale prices still usable?
- Prices that are impossible (0 €, 20 €/L) or look like typos (1.9 vs 19): outliers to triage.
- A missing price for a fuel: does the station not sell it (structural missing) or did it not report?
- The data is a **snapshot**: re-running tomorrow gives different rows. Write that in Section 1.

In [ ]:
decisions = []
def log(column, issue, n_rows, decision, why):
    decisions.append(dict(column=column, issue=issue, n_rows=n_rows, decision=decision, why=why))

df = df_sel.copy()

### 4.1 Duplicates

In [ ]:
n_exact = int(df.astype(str).duplicated().sum())    # astype(str): some columns may hold dicts
print("exact duplicate rows:", n_exact)
# TODO: decide, apply, and log. Then look for near-duplicates (same entity, different id or spelling).
# Near-duplicates: same postal code and same normalised address, but different id
norm = df["adresse"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
key = df["cp"] + "|" + norm
near = df[key.duplicated(keep=False)].assign(key=key).sort_values("key")
print("rows sharing postal code + address with another id:", len(near))
display(near[["id", "cp", "ville", "adresse", "prix_gazole"]].head(20))
# TODO: decide what a near-duplicate is (same station?), apply, log.


### 4.2 Impossible values

In [ ]:
PRICE_COLS = [c for c in df.columns if c.startswith("prix_")]
out_range = {c: int(((df[c] < PRICE_MIN) | (df[c] > PRICE_MAX)).sum()) for c in PRICE_COLS}
print("prices outside [PRICE_MIN, PRICE_MAX], per fuel:", out_range)
bad_geo = int((~df["lat"].between(*FR_LAT) | ~df["lon"].between(*FR_LON)).sum())
print("coordinates outside France:", bad_geo)
future = {c: int((df[c] > pd.Timestamp(collected_at).tz_convert("UTC").tz_localize(None)).sum())
          for c in df.columns if c.startswith("maj_")}
print("update dates in the future:", future)
# TODO: decide for each count: fix, set to NaN, or drop. Then log(...).
df.describe().T

### 4.3 Missingness — mechanism before method

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0].rename("share_missing").to_frame()
miss["flag"] = np.where(miss.share_missing > MAX_MISSING_SHARE, "⚠️ high", "")
miss

In [ ]:
# Key column: prix_sp95 (most missing). Does its missingness depend on region?
rng = np.random.default_rng(SEED)
m = df["prix_sp95"].isna()
by_region = df.assign(m=m).groupby("region")["m"].agg(share_missing="mean", n="size").sort_values("share_missing")
display(by_region)

# Permutation test of the spread of missing shares across regions (numpy only, reproducible via SEED)
def spread(mask, groups):
    shares = pd.Series(mask).groupby(groups.values).mean()
    return shares.max() - shares.min()
obs = spread(m.values, df["region"].fillna("NA"))
perm = np.array([spread(rng.permutation(m.values), df["region"].fillna("NA")) for _ in range(1000)])
print(f"observed spread: {obs:.3f} · permutation p-value: {(perm >= obs).mean():.3f}")
# TODO: name the mechanism (MCAR / MAR / MNAR) from this test and from the 'Carburants' columns, decide, log.

### 4.4 Outliers — triage, don't delete

In [ ]:
# Key numeric column: prix_gazole. IQR fence on the log scale (price is right-skewed)
s = np.log(df["prix_gazole"].dropna())
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = np.exp(q1 - IQR_K * iqr), np.exp(q3 + IQR_K * iqr)
flag = (df["prix_gazole"] < lo) | (df["prix_gazole"] > hi)
print(f"fence: [{lo:.3f}, {hi:.3f}] EUR/L · flagged rows: {int(flag.sum())}")
display(df.loc[flag, ["id", "ville", "prix_gazole"]].head(20))
# TODO: separate errors (typos, impossible) from real extremes (motorway stations). Never drop silently: log each decision.

In [ ]:
decision_table = pd.DataFrame(decisions, columns=["column", "issue", "n_rows", "decision", "why"])
decision_table.to_csv(REPORTS / "decision_table.csv", index=False)
decision_table

---
## 5 · Exploration — every figure gets a caption and a *so what*

In [ ]:
def show(fig, title, caption, so_what, name):
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.text(0.01, -0.04, f"{caption}\n→ {so_what}", ha="left", va="top", fontsize=9, wrap=True)
    fig.savefig(REPORTS / f"{name}.png", bbox_inches="tight", dpi=120)
    plt.show()

clean = df.copy()      # TODO: this should be your cleaned data after Section 4

### 5.1 Univariate

In [ ]:
# TODO: one figure showing the distribution of your key variable (ECDF before `.describe()`)
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_1")

### 5.2 Bivariate

In [ ]:
# TODO: one figure showing your key variable against 1–2 explanatory variables
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_2")

### 5.3 Multivariate

In [ ]:
# TODO: one figure showing the correlation structure, or a breakdown by two categories
# fig, ax = plt.subplots(figsize=(8, 4))
# ...
# show(fig, "<title that states the finding>", "<what the figure shows>", "<what a modeller should do>", "5_3")

---
## 6 · Findings, what cannot be trusted, next steps

*TODO*

**Three findings** — each with the figure it comes from and what it means for modelling.
1.
2.
3.

**What cannot be trusted** — column, subset, and why.
-
-

**Five testable hypotheses** — variable, direction, population, test.
- H1:
- H2:
- H3:
- H4:
- H5:

### Final contract & fingerprint
*TODO*: write `clean_schema` for the data a modeller receives (it may differ from the raw schema). Then **Restart & Run All twice**: the fingerprint must be identical.

In [ ]:
clean_schema = None                  # TODO: pa.DataFrameSchema({...}, strict=True)
if clean_schema is not None:
    clean_schema.validate(clean, lazy=True)
    print("✅ clean data passes the contract")

clean.to_parquet(DATA / "processed" / "clean.parquet", index=False)
fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(clean.astype(str), index=False).values.tobytes()).hexdigest()[:12]
print("rows:", len(clean), "· fingerprint:", fingerprint)

> **Reproducibility note.** The fingerprint is stable between runs *thanks to the cache*: the API data changes over time, so a run without the cache (or after `CACHE_HOURS`) can give a different fingerprint. That is expected — record the collection date in Section 1 and in the README, and commit the raw Parquet if it is small enough.